# Data Acquisition + ETL Plan

**DS2002 | 2026-04-01 | Studio (Hands-On Lab)**

---

## Overview

Today you will:
1. Authenticate to Google Cloud Storage from your Kaggle notebook.
2. List the bucket contents and verify your team folder.
3. Download the raw data files.
4. Inspect the raw data and identify quality issues.
5. Upload a test file to your team folder.
6. Draft your ETL plan.

This is your first cloud lab. Work with your team. If you get stuck on authentication, check the `GCP_Console_Walkthrough.md` guide.

## Part 1 — Install and import

In [ ]:
!pip install google-cloud-storage -q

import os
import json
import pandas as pd
import sqlite3

## Part 2 — Authenticate to GCS

Your team should have received a JSON service account key and added it as a Kaggle Secret named `gcs_key`.

If you have not done this yet, go to Add-ons > Secrets in Kaggle and add it now. See `GCP_Console_Walkthrough.md` Section 5 for instructions.

In [ ]:
from google.cloud import storage
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
key_json = secrets.get_secret("gcs_key")
client = storage.Client.from_service_account_info(json.loads(key_json))
bucket = client.bucket("ds2002-capstone-sp26")

print("Authenticated successfully.")
print(f"Bucket: {bucket.name}")

## Part 3 — List bucket contents

Run this to see what is in the bucket. You should see the `raw-data/` folder and all the team folders.

In [ ]:
blobs = list(bucket.list_blobs())
for b in blobs[:30]:
    print(f"  {b.name}  ({b.size} bytes)")

if len(blobs) > 30:
    print(f"  ... and {len(blobs) - 30} more files")

### Check your team folder

Change `TEAM` to your actual team number.

In [ ]:
TEAM = "team-XX"  # <-- change this

team_blobs = list(bucket.list_blobs(prefix=f"{TEAM}/"))
print(f"Files in {TEAM}/")
for b in team_blobs:
    print(f"  {b.name}")

## Part 4 — Download raw data

Download all 5 raw data files to a local `data/` directory in your notebook environment.

In [ ]:
files = [
    "raw-data/charging_sessions.csv",
    "raw-data/station_locations.csv",
    "raw-data/vehicle_types.csv",
    "raw-data/grid_operators.csv",
    "raw-data/energy_and_demand.db",
]

os.makedirs("data", exist_ok=True)
for f in files:
    blob = bucket.blob(f)
    local = os.path.join("data", os.path.basename(f))
    blob.download_to_filename(local)
    print(f"Downloaded: {f} -> {local}")

print("\nAll files downloaded.")

## Part 5 — Inspect the raw data

Load each file and run the standard inspection: shape, dtypes, nulls, head.

In [ ]:
sessions = pd.read_csv("data/charging_sessions.csv")
print("charging_sessions.csv")
print(f"  Shape: {sessions.shape}")
print(f"  Columns: {sessions.columns.tolist()}")
print(f"  Null counts:")
print(sessions.isnull().sum())
sessions.head()

In [ ]:
stations = pd.read_csv("data/station_locations.csv")
print("station_locations.csv")
print(f"  Shape: {stations.shape}")
stations

In [ ]:
vehicles = pd.read_csv("data/vehicle_types.csv")
print("vehicle_types.csv")
print(f"  Shape: {vehicles.shape}")
vehicles.head(10)

In [ ]:
operators = pd.read_csv("data/grid_operators.csv")
print("grid_operators.csv")
operators

In [ ]:
conn = sqlite3.connect("data/energy_and_demand.db")
demand = pd.read_sql("SELECT * FROM daily_demand_summary LIMIT 5", conn)
grid = pd.read_sql("SELECT * FROM grid_capacity_levels LIMIT 5", conn)
conn.close()

print("daily_demand_summary:")
print(demand)
print("\ngrid_capacity_levels:")
print(grid)

### What do you see?

Before moving on, write down (in a markdown cell or on paper) at least 5 data quality issues you have spotted across these files. Be specific — which file, which column, what is wrong.

In [ ]:
# Your notes on data quality issues:
#
# 1.
# 2.
# 3.
# 4.
# 5.


## Part 6 — Upload a test file

Prove that your write access works. Create a small text file and upload it to your team folder.

In [ ]:
# Create a test file
with open("test_upload.txt", "w") as f:
    f.write("Cloud access confirmed for " + TEAM)

# Upload to GCS
blob = bucket.blob(f"{TEAM}/test_upload.txt")
blob.upload_from_filename("test_upload.txt")
print(f"Uploaded test_upload.txt to gs://ds2002-capstone-sp26/{TEAM}/test_upload.txt")

# Verify
team_blobs = list(bucket.list_blobs(prefix=f"{TEAM}/"))
for b in team_blobs:
    print(f"  {b.name}")

## Part 7 — Draft your ETL plan

Now that you have seen the data, plan your cleaning pipeline. Fill in the table below.

| Issue | File(s) Affected | Cleaning Strategy | Priority |
|-------|-----------------|-------------------|----------|
| Example: duplicate records | charging_sessions.csv | Drop exact duplicates using `.drop_duplicates()` | High |
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |

Bring this plan to the April 5 proposal submission.

---

## Wrap-up

By the end of today you should have:
- Working GCS authentication from Kaggle
- All 5 raw data files downloaded
- A test file uploaded to your team folder
- A written list of at least 5 data quality issues
- A draft ETL plan

If any of these are not done, finish them before Tuesday.